# CelebA: privacy, utility, and participant scaling

This standalone notebook runs the current Noisy Anchor Alignment experiment. Select a **GPU runtime**, then **Run all**. It saves each completed fit to Google Drive and resumes by validating and skipping completed fits. Keep this notebook in **Noisy Anchor Alignment** and replace it in place when revised.

### Fixed experiment budget

| Component | Design | Utility fits |
|---|---|---:|
| NAA-GDP, per participant count | 10 positive scales × 4 realizations × 5 deployment seeds | **200** |
| NAA-GDP, all counts | p = 2, 5, 10, 20, 50 | **1,000** |
| C-GDP with private-data noise | p = 10; 10 positive scales × 4 × 5 | 200 |
| Noiseless C-GDP, I-GDP, AA-GDP | once per method, count, and seed | 75 |
| Local-only MLPs | one per participant and seed, reused across nested counts | 250 |
| **Total** | | **1,525** |

The anchor-noise grid is **0.006, 0.012, …, 0.060**. Zero is evaluated separately through the noiseless controls. The private-data-noise comparator uses σ = 0.3, 0.6, …, 3.0. Both are absolute entrywise noise SDs, not RMS-normalized multipliers. Four realizations mean four independently sampled noise matrices at each positive level within each fixed deployment. They are alternative releases; an attacker never receives all four together.

Each seed creates one fresh 50-participant deployment. Smaller collaborations are nested prefixes. Every main p=10 observation comes directly from that same deployment's participant-count experiment.

### Data and evaluation

- Center-square RGB CelebA images are resized to 64×64 and normalized to [0,1]. A fresh, uncentered public SVD on 10,000 images produces an orthonormal 400-dimensional projection for each seed. Public identities and participant identities are disjoint.
- Each participant has 100 identities, assigned to train/validation/test by identity in a 70/15/15 ratio with image-count and label balancing. One clean reference image per identity is withheld from all uploads.
- Utility is test Smiling balanced accuracy from an MLP **400→256→128→1**, ReLU, Adam, batch size 256, learning rate 0.001. Train for at most **100 epochs**, stop after 10 epochs without improvement, and select the checkpoint using validation balanced accuracy. The training seed is fixed within each deployment across methods and noise conditions. Test data never select the checkpoint.
- **Local-only** trains one such MLP on each participant's own unperturbed public-SVD features, using only that participant's training and validation sets. Its model is evaluated only on its own test set. Pool the local test confusion counts over the first p participants to obtain a balanced accuracy comparable with pooled collaborative utility. Fit each local model once and reuse it across nested participant counts; the 25 local aggregate rows are summaries, not additional fits. Local-only has no data upload and receives no reconstructed-image linkage score; I-GDP is also retained as a utility-only reference.
- Privacy uses the existing **OP reconstruction and frozen VGGFace2 InceptionResnetV1 auditor**. Target participant 1 contributes 100 queries, one uploaded image per identity. Each query is tested in ten fixed ten-candidate lineups, producing 1,000 comparisons. Its correct candidate is a different, reserved image of the same identity.
- The gallery pool is explicitly fixed to the target participant's **100 reserved references** for every p. This avoids changing the identity-matching task as p grows; it differs from legacy code that could draw decoys from all active participants. No direct-matching attack or additional dataset is introduced.
- Honest GPM alignment uses only noisy anchors, float64 arithmetic, spectral initialization, absolute step tolerance 1e-7, and a 500-iteration cap. Its output is expressed in participant 0's reference coordinates. Solver convergence and wall time are recorded for every noisy run; capped runs remain in the results.
- The private-noise attack uses the exact noise in each actual uploaded query record. For NAA, the target upload, noisy target anchor, and gallery are identical across p, so the same OP-auditor result is reused exactly across counts.

### Reporting and reuse

Average four realizations within each seed and noise level, then report the mean and **SD across the five seed means**. The notebook saves the individual seed means as well. Curves connect evaluated levels; no spline extrapolation or chance-crossing claim is imposed. This is an evaluation of the specified reconstruction/auditor pipeline, not a privacy guarantee against all attacks.

Only verified raw CelebA archive, identity metadata, RGB pixels, and public auditor weights may be reused from the separate **NAA-GDP** folder. Old projections, splits, transformations, fitted models, and experimental results are not imported. All experimental implementation is embedded below; no clone or import of the old repository is required. Protocol reference: [anchor-side-perturbation-reproducibility](https://github.com/KU-Nosaka/anchor-side-perturbation-reproducibility).

Results, manifests, generated figures, and derived caches are saved under `results/celeba/`. Restart **Run all** to resume after a runtime interruption. Changes to code or configuration are rejected if incompatible results already exist, preventing silent mixing. A lost in-progress fit restarts; completed fits are preserved. Colab availability determines how long one session can run.


In [ ]:
# Install only the auditor package; keep Colab's matching torch/torchvision builds.
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import subprocess
import sys
import importlib.metadata
try:
    installed = importlib.metadata.version("facenet-pytorch")
except importlib.metadata.PackageNotFoundError:
    installed = None
if installed != "2.6.0":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "facenet-pytorch==2.6.0"])
from google.colab import drive
drive.mount("/content/drive")
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Use Runtime > Change runtime type and select a GPU, then Run all.")
print("Connected GPU:", torch.cuda.get_device_name())


## Configuration
The counts below are fixed for this run. Epochs and GPM iterations are optimization limits, separate from the 200 noisy-anchor experiments per participant count.


In [ ]:
CONFIG = {'seeds': [0, 1, 2, 3, 4], 'realizations': 4, 'anchor_scales': [0.006, 0.012, 0.018, 0.024, 0.03, 0.036, 0.042, 0.048, 0.054, 0.06], 'participant_counts': [2, 5, 10, 20, 50], 'private_sigmas': [0.3, 0.6, 0.9, 1.2, 1.5, 1.8, 2.1, 2.4, 2.7, 3.0], 'h': 400, 'r': 1600, 'target_participant': 1, 'reference_participant': 0, 'gpm_tolerance': 1e-07, 'gpm_max_iters': 500, 'mlp': {'hidden_dims': [256, 128], 'max_epochs': 100, 'patience': 10, 'batch_size': 256, 'lr': 0.001, 'weight_decay': 0.0}, 'old_cache': '/content/drive/MyDrive/NAA-GDP/CelebA_PublicSVD_identity100_cache_v1', 'output_root': '/content/drive/MyDrive/Noisy Anchor Alignment/results/celeba', 'protocol': 'celeba-op-public-auditor-five-seed-v1', 'gallery_pool': '100 reserved references of target participant, fixed across p', 'noise_units': 'absolute entrywise standard deviation in the reduced representation', 'local_baseline': 'one MLP per participant on its own unperturbed public-SVD features; pooled test confusion counts'}
CODE_SHA256 = '76345fd2ebe4d59150e02c134c962ff1b0f395dc8494bfbc97fa2ac6b9b23eab'
print("Anchor levels:", CONFIG["anchor_scales"])
print("NAA runs per participant count:", len(CONFIG["seeds"])*len(CONFIG["anchor_scales"])*CONFIG["realizations"])


## Data preparation
Implementation is included in full so this notebook remains standalone.


In [ ]:
"""Fresh CelebA deployments; old NAA-GDP assets are read-only raw-data inputs."""
from __future__ import annotations

import hashlib
import io
import json
import os
from pathlib import Path
import shutil
import tempfile
import zipfile

import numpy as np
import pandas as pd
from PIL import Image
import torch

DATA_PREPROCESS = "celeba-center-square-rgb64-v1"
DATA_SCHEMA = "naa-celeba-fresh-20260906-v1"


def _data_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":")).encode()).hexdigest()


def _data_seed(seed, *parts):
    return int(_data_hash([int(seed), *parts])[:15], 16)


def _data_sha(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def _data_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(value, indent=2, sort_keys=True) + "\n")
    os.replace(tmp, path)


def _data_order(seed, label, values):
    return sorted(values, key=lambda item: _data_hash([int(seed), label, item]))


def _data_mirror(source, destination, expected):
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.is_file() and _data_sha(destination) == expected:
        return destination
    tmp = destination.with_suffix(destination.suffix + ".tmp")
    shutil.copyfile(source, tmp)
    if _data_sha(tmp) != expected:
        raise RuntimeError(f"Content checksum mismatch: {source}")
    os.replace(tmp, destination)
    return destination


def _data_read_metadata(archive, identity_path):
    with zipfile.ZipFile(archive) as handle:
        matches = [name for name in handle.namelist() if Path(name).name == "list_attr_celeba.csv"]
        if len(matches) != 1:
            raise RuntimeError("Expected one list_attr_celeba.csv in the source archive")
        attr_bytes = handle.read(matches[0])
    identity_bytes = identity_path.read_bytes()
    if identity_path.suffix.lower() == ".txt":
        identity = pd.read_csv(io.BytesIO(identity_bytes), sep=r"\s+", header=None,
                               names=["image_id", "identity"])
    else:
        identity = pd.read_csv(io.BytesIO(identity_bytes))
        identity.columns = ["image_id", "identity"]
    attr = pd.read_csv(io.BytesIO(attr_bytes))
    attr = attr.rename(columns={attr.columns[0]: "image_id"})
    frame = attr[["image_id", "Smiling"]].merge(identity, on="image_id", validate="one_to_one")
    frame = frame.sort_values("image_id").reset_index(drop=True)
    frame["identity"] = frame.identity.astype(np.int64)
    frame["label"] = (frame.Smiling.astype(int) > 0).astype(np.int64)
    if len(frame) != 202599 or frame.identity.nunique() != 10177 or not frame.image_id.is_unique:
        raise RuntimeError("Source must contain all 202,599 images and 10,177 identities")
    return frame, {"attributes": hashlib.sha256(attr_bytes).hexdigest(),
                   "identities": hashlib.sha256(identity_bytes).hexdigest()}


def _data_build_rgb(old_cache, archive, archive_hash, names, local_rgb):
    """Reuse only verified uint8 pixels; generate any missing pixels from raw JPGs."""
    tmp = local_rgb.with_suffix(".building.npy")
    pixels = np.lib.format.open_memmap(tmp, mode="w+", dtype=np.uint8,
                                     shape=(len(names), 3, 64, 64))
    row = {name: i for i, name in enumerate(names)}
    written = np.zeros(len(names), dtype=bool)
    provenance = []
    for path in sorted((old_cache / "rgb64_shards").glob("*/manifest.json")):
        manifest = json.loads(path.read_text())
        if manifest.get("preprocess") != DATA_PREPROCESS:
            continue
        offset = 0
        for record in manifest["shards"]:
            shard_names = manifest["filenames"][offset:offset + int(record["count"])]
            offset += int(record["count"])
            wanted = [(j, row[name]) for j, name in enumerate(shard_names)
                      if name in row and not written[row[name]]]
            if not wanted:
                continue
            shard = path.parent / record["file"]
            if _data_sha(shard) != record["sha256"]:
                raise RuntimeError(f"Old RGB shard checksum mismatch: {shard}")
            payload = torch.load(shard, map_location="cpu", weights_only=True)
            images = payload["images"]
            if list(payload["filenames"]) != shard_names or images.dtype != torch.uint8:
                raise RuntimeError(f"Old RGB shard metadata mismatch: {shard}")
            if tuple(images.shape) != (len(shard_names), 3, 64, 64):
                raise RuntimeError(f"Old RGB shard shape mismatch: {shard}")
            src, dst = zip(*wanted)
            pixels[list(dst)] = images[list(src)].numpy()
            written[list(dst)] = True
            provenance.append({"path": str(shard), "sha256": record["sha256"]})
        if offset != len(manifest["filenames"]):
            raise RuntimeError(f"Incomplete old RGB manifest: {path}")
    print(f"Raw image cache: reused {int(written.sum()):,} verified RGB images", flush=True)
    if not written.all():
        local_archive = _data_mirror(archive, local_rgb.parent / "raw_source.zip", archive_hash)
        with zipfile.ZipFile(local_archive) as handle:
            members = {Path(name).name: name for name in handle.namelist() if name.lower().endswith(".jpg")}
            for count, index in enumerate(np.flatnonzero(~written), 1):
                with Image.open(io.BytesIO(handle.read(members[names[index]]))) as image:
                    image = image.convert("RGB")
                    w, h = image.size
                    side = min(w, h)
                    left, top = (w - side) // 2, (h - side) // 2
                    image = image.crop((left, top, left + side, top + side)).resize(
                        (64, 64), Image.Resampling.BILINEAR)
                    pixels[index] = np.asarray(image, dtype=np.uint8).transpose(2, 0, 1)
                if count % 20000 == 0:
                    print(f"Raw image cache: converted {count:,} additional images", flush=True)
    pixels.flush()
    del pixels
    os.replace(tmp, local_rgb)
    return provenance


def prepare_source(old_cache: Path, new_cache: Path) -> dict:
    """Read old raw archive/metadata/RGB shards; write only to new_cache/runtime tmp."""
    old_cache, new_cache = Path(old_cache).resolve(), Path(new_cache).resolve()
    if old_cache == new_cache or old_cache in new_cache.parents or new_cache in old_cache.parents:
        raise ValueError("Old read-only cache and new cache must be separate directories")
    archive = old_cache / "raw_archive/celeba_kaggle_mirror.zip"
    archive_manifest = old_cache / "raw_archive/celeba_kaggle_mirror_manifest.json"
    if not archive.is_file() or not archive_manifest.is_file():
        raise FileNotFoundError(f"Missing old raw archive or manifest: {archive}")
    old_info = json.loads(archive_manifest.read_text())
    if archive.stat().st_size != int(old_info["bytes"]):
        raise RuntimeError("Old raw archive size differs from its manifest")
    candidates = [p for pattern in ("identity_CelebA.txt", "identity_CelebA.csv",
                                    "list_identity_celeba.txt", "list_identity_celeba.csv")
                  for p in old_cache.rglob(pattern)]
    if not candidates:
        raise FileNotFoundError("Official CelebA identity metadata is missing from the old raw cache")
    identity_path = sorted(candidates)[0]
    frame, metadata_hashes = _data_read_metadata(archive, identity_path)
    spec = {"schema": DATA_SCHEMA, "preprocess": DATA_PREPROCESS,
            "archive_sha256": old_info["sha256"], "metadata_sha256": metadata_hashes,
            "image_order_sha256": _data_hash(frame.image_id.tolist())}
    source_hash = _data_hash(spec)
    persistent = new_cache / "raw_rgb64" / source_hash[:20]
    persistent.mkdir(parents=True, exist_ok=True)
    local = Path(tempfile.gettempdir()) / "naa_celeba_raw" / source_hash[:20]
    local.mkdir(parents=True, exist_ok=True)
    rgb, info_path = persistent / "images.npy", persistent / "manifest.json"
    local_rgb = local / "images.npy"
    if info_path.is_file() and rgb.is_file():
        info = json.loads(info_path.read_text())
        if info["spec"] != spec:
            raise RuntimeError("Raw RGB cache configuration mismatch")
        _data_mirror(rgb, local_rgb, info["rgb_sha256"])
    else:
        reused = _data_build_rgb(old_cache, archive, old_info["sha256"],
                                 frame.image_id.tolist(), local_rgb)
        checksum = _data_sha(local_rgb)
        _data_mirror(local_rgb, rgb, checksum)
        info = {"spec": spec, "source_hash": source_hash, "rgb_sha256": checksum,
                "readonly_inputs": [{"path": str(archive), "sha256": old_info["sha256"]},
                                    {"path": str(identity_path), "sha256": metadata_hashes["identities"]},
                                    {"path": str(archive_manifest), "sha256": _data_sha(archive_manifest)},
                                    *reused]}
        _data_json(info_path, info)
    pixels = np.load(local_rgb, mmap_mode="r")
    if pixels.shape != (len(frame), 3, 64, 64) or pixels.dtype != np.uint8:
        raise RuntimeError("New raw RGB cache has an invalid shape or dtype")
    return {"frame": frame, "pixels": pixels, "source_hash": source_hash,
            "source_manifest": info, "new_cache": new_cache}


def _data_allocate(frame, seed, max_participants):
    groups = {int(identity): group.index.tolist() for identity, group in frame.groupby("identity")}
    eligible = [identity for identity, rows in groups.items() if len(rows) >= 2]
    public_ids, n = [], 0
    for identity in _data_order(seed, "public-identities", eligible):
        public_ids.append(identity)
        n += len(groups[identity])
        if n >= 10000:
            break
    public_set = set(public_ids)
    public_rows = _data_order(seed, "public-images", [i for identity in public_ids for i in groups[identity]])[:10000]
    chosen = _data_order(seed, "participant-identities", [i for i in eligible if i not in public_set])[:max_participants * 100]
    if len(public_rows) != 10000 or len(chosen) != max_participants * 100:
        raise RuntimeError("Insufficient eligible public/participant identities")
    stats = []
    for identity in chosen:
        rows = _data_order(seed, f"identity-{identity}-images", groups[identity])
        stats.append({"identity": identity, "reference_row": rows[0], "rows": rows[1:],
                      "n": len(rows) - 1, "positive": int(frame.loc[rows[1:], "label"].sum())})
    capacities = {(p, split): cap for p in range(max_participants)
                  for split, cap in (("train", 70), ("val", 15), ("test", 15))}
    buckets, loads, positives = ({k: [] for k in capacities}, {k: 0 for k in capacities}, {k: 0 for k in capacities})
    mean_n = sum(s["n"] for s in stats) / len(stats)
    prevalence = sum(s["positive"] for s in stats) / sum(s["n"] for s in stats)
    stats.sort(key=lambda s: (-s["n"], -abs(s["positive"] / s["n"] - prevalence),
                              _data_hash([seed, "allocation", s["identity"]])))
    for item in stats:
        def score(key):
            target = mean_n * capacities[key]
            # Minimize the CHANGE in the global bucket cost. Comparing absolute
            # post-assignment costs instead would preferentially fill one bucket.
            def cost(n, positive):
                return (n - target) ** 2 / target + .2 * (positive - prevalence * n) ** 2 / max(1, target * prevalence * (1 - prevalence))
            delta = cost(loads[key] + item["n"], positives[key] + item["positive"]) - cost(loads[key], positives[key])
            return delta + .001 * (len(buckets[key]) + 1) / capacities[key], _data_hash([seed, item["identity"], key])
        key = min((k for k in capacities if len(buckets[k]) < capacities[k]), key=score)
        buckets[key].append(item)
        loads[key] += item["n"]
        positives[key] += item["positive"]
    assignments = [{"participant": p, "split": split, **item}
                   for (p, split), items in sorted(buckets.items())
                   for item in sorted(items, key=lambda item: item["identity"])]
    refs = {item["reference_row"] for item in assignments}
    uploads = {i for item in assignments for i in item["rows"]}
    assert len({item["identity"] for item in assignments}) == max_participants * 100
    assert not refs & uploads and not public_set & set(chosen)
    assert all(len(buckets[k]) == cap for k, cap in capacities.items())
    return public_ids, public_rows, assignments


def prepare_deployment(source, seed: int, outdir: Path, h=400, max_participants=50, device=None):
    """Return CPU tensors; refit public SVD and participant splits for every outer seed."""
    device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    frame, pixels = source["frame"], source["pixels"]
    public_ids, public_rows, assignments = _data_allocate(frame, seed, max_participants)
    spec = {"schema": DATA_SCHEMA, "source_hash": source["source_hash"], "seed": int(seed),
            "h": h, "max_participants": max_participants, "public_images": 10000,
            "uncentered": True, "svd_q": h + 20, "svd_niter": 4,
            "target_participant": 1, "colluder_participant": 0,
            "gallery_pool": "target participant's 100 reserved clean references, fixed across p",
            "public_identity_ids": public_ids, "public_rows": public_rows,
            "assignments": assignments}
    config_hash = _data_hash(spec)
    cache = source["new_cache"] / "deployments" / config_hash[:20]
    cache.mkdir(parents=True, exist_ok=True)
    data_path, manifest_path = cache / "deployment.pt", cache / "manifest.json"
    if data_path.is_file() and manifest_path.is_file():
        manifest = json.loads(manifest_path.read_text())
        if manifest["config_hash"] != config_hash or _data_sha(data_path) != manifest["data_sha256"]:
            raise RuntimeError("Deployment cache failed configuration/content verification")
        result = torch.load(data_path, map_location="cpu", weights_only=True)
        result["manifest"] = manifest
        _data_json(outdir / "deployment_manifest.json", manifest)
        return result
    print(f"Seed {seed}: fitting fresh public SVD on 10,000 images", flush=True)
    matrix = torch.from_numpy(np.array(pixels[public_rows])).flatten(1).to(device, torch.float32).div_(255)
    with torch.random.fork_rng(devices=[device.index or 0] if device.type == "cuda" else []):
        torch.manual_seed(_data_seed(seed, "public-svd"))
        _, singular, candidate = torch.pca_lowrank(matrix, q=h + 20, center=False, niter=4)
    F = torch.linalg.qr(candidate[:, :h].double().cpu(), mode="reduced")[0].float()
    residual = float(torch.linalg.matrix_norm(F.double().T @ F.double() - torch.eye(h)))
    if residual > 1e-3:
        raise RuntimeError(f"Projection orthogonality check failed: {residual}")
    del matrix, candidate
    basis = F.to(device)
    def project(rows):
        blocks = []
        for start in range(0, len(rows), 512):
            batch = torch.from_numpy(np.array(pixels[rows[start:start + 512]])).flatten(1).to(device, torch.float32).div_(255)
            blocks.append((batch @ basis).cpu())
        return torch.cat(blocks)
    participants, samples = [], []
    queries, query_rows, query_ids, query_locations, refs = [], [], [], [], []
    for p in range(max_participants):
        participant = {}
        for split in ("train", "val", "test"):
            items = [a for a in assignments if a["participant"] == p and a["split"] == split]
            rows = [i for a in items for i in a["rows"]]
            q = project(rows)
            participant[f"{split}_q"] = q
            participant[f"{split}_y"] = torch.tensor(frame.loc[rows, "label"].to_numpy(), dtype=torch.float32)
            participant[f"{split}_identities"] = torch.tensor(frame.loc[rows, "identity"].to_numpy())
            samples.append({"participant": p, "split": split, "images": len(rows),
                            "identities": len(items), "smiling": int(frame.loc[rows, "label"].sum())})
            if p == 1:
                lookup = {row: j for j, row in enumerate(rows)}
                for a in items:
                    row = _data_order(seed, f"query-{a['identity']}", a["rows"])[0]
                    queries.append(q[lookup[row]])
                    query_rows.append(row)
                    query_ids.append(a["identity"])
                    query_locations.append({"split": split, "position": lookup[row]})
                    refs.append(a["reference_row"])
        participants.append(participant)
        print(f"Seed {seed}: projected participant {p + 1}/{max_participants}", flush=True)
    rng = np.random.default_rng(_data_seed(seed, "fixed-target-gallery"))
    lineups = np.empty((100, 10, 10), dtype=np.int64)
    correct = np.empty((100, 10), dtype=np.int64)
    for i in range(100):
        for episode in range(10):
            candidates = np.r_[i, rng.choice(np.delete(np.arange(100), i), 9, replace=False)]
            rng.shuffle(candidates)
            lineups[i, episode], correct[i, episode] = candidates, int(np.flatnonzero(candidates == i)[0])
    assert len(queries) == 100 and not set(query_rows) & set(refs)
    result = {"participants": participants, "F": F, "query_q": torch.stack(queries),
              "gallery_images": torch.from_numpy(np.array(pixels[refs])).float().div_(255),
              "query_images": torch.from_numpy(np.array(pixels[query_rows])).float().div_(255),
              "lineup_indices": torch.from_numpy(lineups), "correct_positions": torch.from_numpy(correct),
              "query_locations": query_locations, "query_identities": query_ids}
    tmp = data_path.with_suffix(".tmp")
    torch.save(result, tmp)
    os.replace(tmp, data_path)
    manifest = {"config_hash": config_hash, "spec": spec, "sample_counts": samples,
                "query_rows": query_rows, "reference_rows": refs, "query_identities": query_ids,
                "lineups": lineups.tolist(), "correct_positions": correct.tolist(),
                "projection_orthogonality_residual": residual,
                "projection_sha256": hashlib.sha256(F.numpy().tobytes()).hexdigest(),
                "retained_singular_values": singular[:h].cpu().tolist(),
                "data_sha256": _data_sha(data_path), "data_path": str(data_path),
                "source_manifest": source["source_manifest"]}
    _data_json(manifest_path, manifest)
    _data_json(outdir / "deployment_manifest.json", manifest)
    result["manifest"] = manifest
    return result


## Transforms, utility MLP, and frozen auditor
Implementation is included in full so this notebook remains standalone.


In [ ]:
"""Standalone CelebA protocol, MLP, and frozen reconstruction-auditor component.

Records are row vectors. Protocol arithmetic is float64; utility and face models
use float32. Rates returned by this module are fractions, not percentages.
"""

import hashlib
import json
import os
import random
import time
from pathlib import Path

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import numpy as np
import torch
from torch import nn
from torch.nn import functional as torch_functional


def _model_seed(seed, *parts):
    payload = json.dumps([int(seed), *parts], separators=(",", ":"), ensure_ascii=True)
    return int.from_bytes(hashlib.sha256(payload.encode()).digest()[:4], "little")


def _model_numpy(value, dtype=np.float64):
    if torch.is_tensor(value):
        value = value.detach().cpu().numpy()
    return np.asarray(value, dtype=dtype)


def _model_sync(device):
    if torch.device(device).type == "cuda":
        torch.cuda.synchronize(device)


def _model_haar(rng, h):
    q, r = np.linalg.qr(rng.standard_normal((h, h)))
    signs = np.where(np.diag(r) >= 0.0, 1.0, -1.0)
    return q * signs[None, :]


def generate_bank(seed, pmax=50, h=400, r=1600):
    """Stable participant-indexed draws; changing pmax cannot change a prefix.

    A is centered and satisfies A.T @ A = I. O_i are independent Haar draws
    on O(h); translations have independent N(0,1) coordinates.
    """
    if not (1 <= h < r and pmax >= 1):
        raise ValueError("Require 1 <= h < r and pmax >= 1.")
    rng = np.random.default_rng(_model_seed(seed, "anchor"))
    a = rng.standard_normal((r, h))
    a -= a.mean(axis=0)
    a, triangular = np.linalg.qr(a, mode="reduced")
    a *= np.where(np.diag(triangular) >= 0, 1.0, -1.0)[None, :]
    rotations, translations = [], []
    for participant in range(pmax):
        rotations.append(_model_haar(np.random.default_rng(
            _model_seed(seed, "rotation", participant)), h))
        translations.append(np.random.default_rng(
            _model_seed(seed, "translation", participant)).standard_normal(h))
    return {"A": a, "O": np.stack(rotations), "psi": np.stack(translations),
            "seed": int(seed), "h": int(h), "r": int(r), "pmax": int(pmax)}


def noisy_anchor_bank(bank, seed, level_index, realization, v):
    """One alternative release. Noise is iid over entries and participants.

    A level/realization's participant prefix is independent of requested p.
    This does not center W before release: its mean perturbs the translation.
    """
    if not np.isfinite(v) or v < 0:
        raise ValueError("Anchor scale must be finite and nonnegative.")
    if "clean_anchors" not in bank:
        bank["clean_anchors"] = [bank["A"] @ o + psi
                                 for o, psi in zip(bank["O"], bank["psi"])]
    if v == 0:
        return [b.copy() for b in bank["clean_anchors"]]
    return [b + float(v) * np.random.default_rng(_model_seed(
        seed, "anchor-noise", int(level_index), int(realization), participant
    )).standard_normal(b.shape)
        for participant, b in enumerate(bank["clean_anchors"])]


def upload_blocks(blocks, bank):
    """Create private-data uploads without perturbation, Y_i = Q_i O_i + psi_i."""
    return [_model_numpy(q) @ bank["O"][i] + bank["psi"][i]
            for i, q in enumerate(blocks)]


def _model_polar(matrix):
    u, _, vh = torch.linalg.svd(matrix, full_matrices=False)
    return u @ vh


def align_anchors(B, device="cuda", tol=1e-7, max_iters=500, reference=0):
    """Ling spectral initialization and unregularized generalized power method.

    No (p*h)-square Gram is allocated. If r < p*h, spectral initialization
    diagonalizes D D.T and recovers right singular vectors as D.T U / s.
    Iteration is exactly R_next = block_polar(D.T D R), as in the old code.
    Stopping uses the OLD absolute Frobenius step <= tol (not relative step).
    The common output gauge is normalized to the reference participant.

    Returns (maps, diagnostics), with maps['rotations_R'] of shape (p,h,h)
    and maps['means_b'] of shape (p,h). Aligned data are
    (Y_i-b_i) @ maps['rotations_R'][i] + b_reference.
    """
    dev = torch.device(device)
    if not B or max_iters < 1 or tol <= 0 or not 0 <= reference < len(B):
        raise ValueError("Invalid anchors, reference, or solver settings.")
    _model_sync(dev)
    started = time.perf_counter()
    blocks = [torch.as_tensor(b, dtype=torch.float64, device=dev) for b in B]
    r, h = blocks[0].shape
    p = len(blocks)
    if any(tuple(b.shape) != (r, h) for b in blocks) or r <= h:
        raise ValueError("Every anchor block must have common shape r x h, with r > h.")
    means = torch.stack([b.mean(dim=0) for b in blocks])
    D = torch.cat([b - b.mean(dim=0) for b in blocks], dim=1)
    del blocks
    if r < p * h:
        left_gram = D @ D.T
        eigenvalues, eigenvectors = torch.linalg.eigh(left_gram)
        eigenvalues = eigenvalues.flip(0)
        top = eigenvalues[:h]
        if float(top[-1].item()) <= torch.finfo(torch.float64).eps * float(top[0].item()):
            raise RuntimeError("Spectral initialization has a numerically deficient top-h subspace.")
        right = (D.T @ eigenvectors[:, -h:].flip(1)) / top.sqrt().unsqueeze(0)
        gap = float((eigenvalues[h - 1] - eigenvalues[h]).item())
        scale = float(top[-1].item())
        del eigenvectors, eigenvalues, left_gram
    else:
        _, singular, vh = torch.linalg.svd(D, full_matrices=False)
        right = vh[:h].T
        scale = float(singular[h - 1].square().item())
        next_value = float(singular[h].square().item()) if len(singular) > h else 0.0
        gap = scale - next_value
        del vh, singular
    stacked = _model_polar(right.reshape(p, h, h)).reshape(p * h, h)
    del right
    _model_sync(dev)
    initialization_seconds = time.perf_counter() - started
    objective_history = [float((D @ stacked).square().sum().item())]
    step_history = []
    converged = False
    iteration_started = time.perf_counter()
    for _ in range(max_iters):
        next_stacked = _model_polar((D.T @ (D @ stacked)).reshape(p, h, h)).reshape(p*h, h)
        step = float(torch.linalg.vector_norm(next_stacked - stacked).item())
        step_history.append(step)
        stacked = next_stacked
        objective_history.append(float((D @ stacked).square().sum().item()))
        if step <= tol:
            converged = True
            break
    _model_sync(dev)
    iteration_seconds = time.perf_counter() - iteration_started
    next_stacked = _model_polar((D.T @ (D @ stacked)).reshape(p, h, h)).reshape(p*h, h)
    fixed_absolute = float(torch.linalg.vector_norm(next_stacked - stacked).item())
    fixed_relative = fixed_absolute / max(1.0, float(torch.linalg.vector_norm(stacked).item()))
    rotations = stacked.reshape(p, h, h)
    # This exact gauge normalization is also applied to the aligned mean.
    rotations = rotations @ rotations[reference].T.clone()
    rotations[reference] = torch.eye(h, device=dev, dtype=torch.float64)
    template = sum((D[:, i*h:(i+1)*h] @ rotations[i] for i in range(p))) / p
    residual = sum(float((D[:, i*h:(i+1)*h] @ rotations[i] - template).square().sum().item())
                   for i in range(p))
    denominator = float(D.square().sum().item())
    maps = {"rotations_R": rotations.detach().cpu().numpy(),
            "means_b": means.detach().cpu().numpy(), "reference": int(reference)}
    _model_sync(dev)
    diagnostics = {
        "algorithm": "spectral-unregularized-GPM",
        "dtype": "float64", "max_iters": int(max_iters), "tolerance": float(tol),
        "stopping_rule": "absolute Frobenius norm of successive stacked iterates",
        "iterations": len(step_history), "converged": bool(converged),
        "termination_reason": "step_tolerance" if converged else "iteration_cap",
        "last_step": step_history[-1], "fixed_point_residual": fixed_relative,
        "fixed_point_residual_absolute": fixed_absolute,
        "gopp_residual": residual, "relative_gopp_residual": residual / max(denominator, 1e-300),
        "eigengap": gap, "relative_eigengap": gap / max(scale, 1e-300),
        "objective_history": objective_history, "step_history": step_history,
        "initialization_seconds": initialization_seconds, "iteration_seconds": iteration_seconds,
        "elapsed_seconds": time.perf_counter() - started,
        "reference_participant": int(reference), "participant_count": int(p),
    }
    return maps, diagnostics


def apply_maps(uploads, maps):
    means, rotations = maps["means_b"], maps["rotations_R"]
    reference = maps.get("reference", 0)
    return [(_model_numpy(y) - means[i]) @ rotations[i] + means[reference]
            for i, y in enumerate(uploads)]


def op_reconstruct(A, B, Y, device="cuda"):
    """Known-anchor OP recovery of projected records; no image decoder here."""
    dev = torch.device(device)
    a = torch.as_tensor(A, device=dev, dtype=torch.float64)
    b = torch.as_tensor(B, device=dev, dtype=torch.float64)
    y = torch.as_tensor(Y, device=dev, dtype=torch.float64)
    am, bm = a.mean(0), b.mean(0)
    rotation = _model_polar((a - am).T @ (b - bm))
    translation = bm - am @ rotation
    return ((y - translation) @ rotation.T).cpu().numpy()


def _model_set_seed(seed):
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    random.seed(int(seed))
    np.random.seed(int(seed) % (2**32))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def _model_metrics(logits, labels):
    prediction = np.asarray(logits).reshape(-1) >= 0.0
    truth = np.asarray(labels).reshape(-1).astype(bool)
    counts = [int((truth == label).sum()) for label in (False, True)]
    if not all(counts):
        raise ValueError("Balanced accuracy requires both classes in the evaluation split.")
    recall = [float((prediction[truth == label] == label).mean()) for label in (False, True)]
    return {"balanced_accuracy": float(np.mean(recall)),
            "accuracy": float((prediction == truth).mean()),
            "recall_not_smiling": recall[0], "recall_smiling": recall[1],
            "n_records": int(len(truth))}


def fit_mlp(trainX, trainY, valX, valY, testX, testY, seed, cfg=None):
    """Fit the fixed vector MLP and select the epoch using validation BA only.

    seed must be held fixed within a deployment for paired protocol/noise runs.
    A fresh local permutation generator is reset for each call. No feature-wise
    normalization or test-dependent threshold selection is performed.
    """
    config = {"device": "cuda" if torch.cuda.is_available() else "cpu",
              "max_epochs": 50, "patience": 8, "batch_size": 256,
              "lr": 1e-3, "weight_decay": 0.0, "hidden_dims": (256, 128)}
    if cfg is not None:
        config.update(cfg if isinstance(cfg, dict) else vars(cfg))
    if int(config["max_epochs"]) < 1 or int(config["patience"]) < 1:
        raise ValueError("max_epochs and patience must be positive.")
    _model_set_seed(seed)
    dev = torch.device(config["device"])
    arrays = [_model_numpy(x, np.float32) for x in (trainX, valX, testX)]
    labels = [_model_numpy(y, np.float32).reshape(-1) for y in (trainY, valY, testY)]
    h = arrays[0].shape[1]
    if any(x.ndim != 2 or x.shape[1] != h or len(x) != len(y)
           or not np.isfinite(x).all() or not np.isin(y, [0.0, 1.0]).all()
           for x, y in zip(arrays, labels)):
        raise ValueError("MLP requires finite n x h inputs and matching binary 0/1 labels.")
    for y in labels:
        if len(np.unique(y)) != 2:
            raise ValueError("Each utility split must contain both smiling classes.")
    hidden = tuple(config["hidden_dims"])
    if len(hidden) != 2:
        raise ValueError("Expected exactly two MLP hidden layers.")
    network = nn.Sequential(nn.Linear(h, hidden[0]), nn.ReLU(),
                            nn.Linear(hidden[0], hidden[1]), nn.ReLU(),
                            nn.Linear(hidden[1], 1)).to(dev)
    optimizer = torch.optim.Adam(network.parameters(), lr=float(config["lr"]),
                                 weight_decay=float(config["weight_decay"]))
    criterion = nn.BCEWithLogitsLoss()
    xtrain = torch.from_numpy(np.ascontiguousarray(arrays[0]))
    ytrain = torch.from_numpy(labels[0].copy())
    permutation_generator = torch.Generator(device="cpu").manual_seed(int(seed))
    batch_size = int(config["batch_size"])

    def predict(x):
        network.eval()
        output = []
        with torch.no_grad():
            for start in range(0, len(x), batch_size):
                batch = torch.as_tensor(x[start:start+batch_size], device=dev, dtype=torch.float32)
                output.append(network(batch).reshape(-1).cpu().numpy())
        return np.concatenate(output)

    _model_sync(dev)
    started = time.perf_counter()
    best_score, best_state, best_epoch, stale = -np.inf, None, 0, 0
    history = []
    for epoch in range(1, int(config["max_epochs"]) + 1):
        network.train()
        permutation = torch.randperm(len(xtrain), generator=permutation_generator)
        loss_sum = 0.0
        for start in range(0, len(xtrain), batch_size):
            indices = permutation[start:start+batch_size]
            xb, yb = xtrain[indices].to(dev), ytrain[indices].to(dev)
            optimizer.zero_grad(set_to_none=True)
            logits = network(xb).reshape(-1)
            loss = criterion(logits, yb)
            if not torch.isfinite(loss):
                raise RuntimeError("Nonfinite MLP training loss.")
            loss.backward()
            optimizer.step()
            loss_sum += float(loss.detach().item()) * len(indices)
        val_metrics = _model_metrics(predict(arrays[1]), labels[1])
        score = val_metrics["balanced_accuracy"]
        history.append({"epoch": epoch, "train_loss": loss_sum / len(xtrain),
                        "validation_balanced_accuracy": score})
        if score > best_score:
            best_score, best_epoch, stale = score, epoch, 0
            best_state = {name: value.detach().cpu().clone()
                          for name, value in network.state_dict().items()}
        else:
            stale += 1
        if stale >= int(config["patience"]):
            break
    network.load_state_dict(best_state)
    # Test records are evaluated exactly once, after the validation selection.
    result = _model_metrics(predict(arrays[2]), labels[2])
    _model_sync(dev)
    result.update({"val_best_balanced_accuracy": float(best_score),
                   "best_epoch": int(best_epoch), "epochs": int(epoch),
                   "train_seconds": time.perf_counter() - started,
                   "training_seed": int(seed), "decision_threshold_logit": 0.0,
                   "architecture": [h, *hidden, 1], "training_history": history,
                   "stopped_early": bool(epoch < int(config["max_epochs"]))})
    del network, optimizer, best_state
    return result


class FaceAuditor:
    """Existing frozen VGGFace2 InceptionResnetV1 cross-image identity auditor."""

    def __init__(self, device="cuda", torchcache=None, batch_size=64):
        if torchcache is not None:
            Path(torchcache).mkdir(parents=True, exist_ok=True)
            os.environ["TORCH_HOME"] = str(torchcache)
        from facenet_pytorch import InceptionResnetV1
        self.device = torch.device(device)
        self.batch_size = int(batch_size)
        self.model = InceptionResnetV1(pretrained="vggface2").eval().to(self.device)
        for parameter in self.model.parameters():
            parameter.requires_grad_(False)
        digest = hashlib.sha256()
        for name, value in sorted(self.model.state_dict().items()):
            digest.update(name.encode())
            digest.update(value.detach().cpu().contiguous().numpy().tobytes())
        self.metadata = {
            "architecture": "InceptionResnetV1", "pretrained": "vggface2",
            "frozen": True, "state_sha256": digest.hexdigest(),
            "preprocessing": "RGB [0,1], bilinear 160x160, (255*x-127.5)/128",
            "comparison": "cosine similarity to clean cross-image references",
        }

    @torch.no_grad()
    def encode_gallery(self, images):
        """Accept uint8 or [0,1] float RGB, NCHW or NHWC, return n x 512 numpy."""
        tensor = torch.as_tensor(images)
        if tensor.ndim != 4:
            raise ValueError("Expected a four-dimensional RGB image batch.")
        if tensor.shape[1] != 3 and tensor.shape[-1] == 3:
            tensor = tensor.permute(0, 3, 1, 2)
        if tensor.shape[1] != 3:
            raise ValueError("Auditor images must have three RGB channels.")
        uint8 = tensor.dtype == torch.uint8
        embeddings = []
        for start in range(0, len(tensor), self.batch_size):
            batch = tensor[start:start+self.batch_size].to(self.device, dtype=torch.float32)
            if uint8:
                batch = batch / 255.0
            if not torch.isfinite(batch).all() or float(batch.min()) < -1e-6 or float(batch.max()) > 1.000001:
                raise ValueError("Auditor floating images must be finite and in [0,1].")
            batch = torch_functional.interpolate(batch, size=(160, 160), mode="bilinear", align_corners=False)
            batch = (batch * 255.0 - 127.5) / 128.0
            embeddings.append(torch_functional.normalize(self.model(batch), dim=1).cpu())
        if not embeddings:
            return np.empty((0, 512), dtype=np.float32)
        return torch.cat(embeddings).numpy()

    @torch.no_grad()
    def linkage(self, qhat, F, gallery_embeddings, lineup_indices, correct_pos):
        """Decode an orthonormal public-SVD representation and score fixed lineups.

        qhat: n_query x h; F: (3*64*64) x h, using CHW flattening.
        lineups: n_query x n_repeated_galleries x n_candidates (or n_query x k).
        Returns identity-balanced linkage when each query represents one identity.
        """
        query = _model_numpy(qhat, np.float32)
        basis = torch.as_tensor(F, dtype=torch.float32, device=self.device)
        if basis.shape[0] != 3 * 64 * 64 or basis.shape[1] != query.shape[1]:
            raise ValueError("F and qhat must use the public RGB64 SVD projection.")
        query_embeddings = []
        for start in range(0, len(query), self.batch_size):
            reduced = torch.as_tensor(query[start:start+self.batch_size], device=self.device)
            decoded = (reduced @ basis.T).clamp(0.0, 1.0).reshape(-1, 3, 64, 64)
            query_embeddings.append(self.encode_gallery(decoded))
        query_embeddings = np.concatenate(query_embeddings)
        gallery = _model_numpy(gallery_embeddings, np.float32)
        indices = np.asarray(lineup_indices, dtype=np.int64)
        truth = np.asarray(correct_pos, dtype=np.int64)
        if indices.ndim == 2:
            indices, truth = indices[:, None, :], truth.reshape(-1, 1)
        if indices.ndim != 3 or len(indices) != len(query) or truth.shape != indices.shape[:2]:
            raise ValueError("Lineup indices and correct positions do not match query count.")
        if (indices < 0).any() or (indices >= len(gallery)).any():
            raise ValueError("Gallery index outside available reference embeddings.")
        if (truth < 0).any() or (truth >= indices.shape[-1]).any():
            raise ValueError("Correct position outside candidate lineup.")
        scores = np.einsum("qd,qgkd->qgk", query_embeddings, gallery[indices])
        success = scores.argmax(axis=-1) == truth
        return {"linkage_rate": float(success.mean()),
                "per_query_success": success.mean(axis=1).tolist(),
                "n_queries": int(len(query)), "lineups_per_query": int(indices.shape[1]),
                "n_candidates": int(indices.shape[2]), "n_lineups": int(success.size),
                "chance_rate": 1.0 / indices.shape[2]}


def protocol_self_test():
    """Small CPU algebra and reference-gauge checks, without pretrained weights."""
    bank = generate_bank(173, pmax=3, h=5, r=17)
    short = generate_bank(173, pmax=2, h=5, r=17)
    np.testing.assert_array_equal(bank["O"][:2], short["O"])
    np.testing.assert_array_equal(bank["A"], short["A"])
    np.testing.assert_allclose(bank["A"].mean(0), 0, atol=2e-15)
    np.testing.assert_allclose(bank["A"].T @ bank["A"], np.eye(5), atol=2e-14)
    data = [np.random.default_rng(i).normal(size=(11, 5)) for i in range(3)]
    uploads = upload_blocks(data, bank)
    clean = noisy_anchor_bank(bank, 173, 0, 0, 0.0)
    maps, diag = align_anchors(clean, device="cpu", tol=1e-10, max_iters=100)
    aligned = apply_maps(uploads, maps)
    for i, result in enumerate(aligned):
        np.testing.assert_allclose(result, data[i] @ bank["O"][0] + bank["psi"][0], atol=1e-11)
        recovered = op_reconstruct(bank["A"], clean[i], uploads[i], device="cpu")
        np.testing.assert_allclose(recovered, data[i], atol=1e-11)
    # Coupling isotropic private noise by orthogonal rotation yields the exact
    # common-reference noisy matrix asserted by the equivalence result.
    errors = [0.1 * np.random.default_rng(500+i).normal(size=q.shape)
              for i, q in enumerate(data)]
    noisy_uploads = [y + errors[i] @ bank["O"][i] for i, y in enumerate(uploads)]
    noisy_aligned = apply_maps(noisy_uploads, maps)
    for i, result in enumerate(noisy_aligned):
        common = (data[i] + errors[i]) @ bank["O"][0] + bank["psi"][0]
        np.testing.assert_allclose(result, common, atol=1e-11)
    np.testing.assert_array_equal(maps["rotations_R"][0], np.eye(5))
    # Common right multiplication must disappear in reference normalization.
    gauge = _model_haar(np.random.default_rng(77), 5)
    raw = maps["rotations_R"] @ gauge
    normalized = raw @ raw[0].T
    np.testing.assert_allclose(normalized, maps["rotations_R"], atol=1e-13)
    noise1 = noisy_anchor_bank(bank, 173, 3, 2, 0.024)
    noise2 = noisy_anchor_bank(short, 173, 3, 2, 0.024)
    for a, b in zip(noise1, noise2):
        np.testing.assert_array_equal(a, b)
    # Exercise the r-square spectral route too (p*h > r).
    wide = generate_bank(31, pmax=5, h=5, r=17)
    _, wide_diag = align_anchors(noisy_anchor_bank(wide, 31, 0, 0, 0), device="cpu",
                                tol=1e-10, max_iters=100)
    assert diag["converged"] and wide_diag["converged"]
    return {"passed": True, "noiseless_gopp_relative_residual": diag["relative_gopp_residual"],
            "wide_spectral_relative_residual": wide_diag["relative_gopp_residual"]}


## Execution and reporting
Implementation is included in full so this notebook remains standalone.


In [ ]:
"""Resumable schedule and seed-level reporting for the standalone notebook."""
import csv
import gc
import importlib.metadata
import platform
import traceback
from datetime import datetime, timezone

SPLITS = ("train", "val", "test")


def _run_completed_counts(root):
    """Count actual fitted models separately from reused local aggregate rows."""
    root = Path(root)
    record_paths = list((root / "records").glob("*.json"))
    aggregate_count = sum("_Local-only_" in path.stem for path in record_paths)
    local_count = sum(1 for _ in (root / "local_models").glob("*.json"))
    return {"completed_fits": len(record_paths) - aggregate_count + local_count,
            "completed_result_rows": len(record_paths),
            "completed_local_fits": local_count,
            "completed_local_aggregates": aggregate_count}


def _run_local_counts(metrics, labels):
    """Recover integer confusion counts from the exact per-class recalls."""
    truth = np.asarray(labels).reshape(-1)
    negative, positive = int((truth == 0).sum()), int((truth == 1).sum())
    if negative + positive != len(truth) or not negative or not positive:
        raise ValueError("Local test labels must contain both binary classes.")
    tn = int(round(metrics["recall_not_smiling"] * negative))
    tp = int(round(metrics["recall_smiling"] * positive))
    if not (0 <= tn <= negative and 0 <= tp <= positive):
        raise RuntimeError("Local confusion counts are inconsistent with test labels.")
    return {"true_negatives": tn, "false_positives": negative - tn,
            "false_negatives": positive - tp, "true_positives": tp}


def _run_pool_local(local_rows):
    """Evaluate all local models' test predictions as one pooled cohort."""
    names = ("true_negatives", "false_positives", "false_negatives", "true_positives")
    counts = {name: sum(row["confusion_counts"][name] for row in local_rows) for name in names}
    tn, fp, fn, tp = (counts[name] for name in names)
    negative, positive = tn + fp, tp + fn
    if negative == 0 or positive == 0:
        raise RuntimeError("Pooled local evaluation requires both classes.")
    return {"balanced_accuracy": 0.5 * (tn / negative + tp / positive),
            "accuracy": (tn + tp) / (negative + positive),
            "recall_not_smiling": tn / negative, "recall_smiling": tp / positive,
            "n_records": negative + positive, "n_local_models": len(local_rows),
            "confusion_counts": counts,
            "aggregation": "pool confusion counts across the participating local models"}


def _run_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2, sort_keys=True, allow_nan=False) + "\n")
    os.replace(temporary, path)


def _run_matmul(x, rotation, shift, device):
    answer = []
    r = torch.as_tensor(rotation, device=device, dtype=torch.float64)
    b = torch.as_tensor(shift, device=device, dtype=torch.float64)
    for start in range(0, len(x), 4096):
        t = torch.as_tensor(x[start:start+4096], device=device, dtype=torch.float64)
        answer.append((t @ r + b).cpu().numpy())
    return np.concatenate(answer)


def _run_collaboration(uploads, maps, p, device):
    result = {}
    for split in SPLITS:
        if maps is None:
            result[split] = np.concatenate(uploads[split][:p]).astype(np.float32)
        else:
            rotations, means = maps["rotations_R"], maps["means_b"]
            reference = maps["reference"]
            result[split] = np.concatenate([
                _run_matmul(uploads[split][i], rotations[i],
                            means[reference] - means[i] @ rotations[i], device).astype(np.float32)
                for i in range(p)])
    return result


def _run_clean_maps(anchors, device):
    b = torch.as_tensor(np.stack(anchors), dtype=torch.float64, device=device)
    means = b.mean(1)
    centered = b - means[:, None, :]
    rotations = _model_polar(centered.transpose(1, 2) @ centered[0])
    rotations[0] = torch.eye(rotations.shape[-1], device=device, dtype=torch.float64)
    return {"rotations_R": rotations.cpu().numpy(), "means_b": means.cpu().numpy(), "reference": 0}


def summarize_results(root, config):
    """First average the four alternative releases, then summarize five deployments."""
    root = Path(root)
    rows = [json.loads(path.read_text()) for path in sorted((root / "records").glob("*.json"))]
    if not rows:
        return pd.DataFrame()
    flat = []
    for row in rows:
        flat.append({key: value for key, value in row.items()
                     if value is None or isinstance(value, (str, int, float, bool))})
    raw = pd.DataFrame(flat)
    raw.to_csv(root / "raw_results.csv", index=False)
    keys = ["method", "participant_count", "noise_scale", "outer_seed"]
    metrics = ["balanced_accuracy", "linkage_rate", "fit_seconds", "alignment_seconds"]
    seed_means = raw.groupby(keys, dropna=False)[metrics].mean().reset_index()
    repetitions = raw.groupby(keys).size().rename("completed_realizations").reset_index()
    seed_means = seed_means.merge(repetitions, on=keys)
    seed_means["required_realizations"] = np.where(seed_means.noise_scale > 0, config["realizations"], 1)
    seed_means["seed_cell_complete"] = seed_means.completed_realizations == seed_means.required_realizations
    seed_means.to_csv(root / "seed_means.csv", index=False)
    complete = seed_means[seed_means.seed_cell_complete]
    summary_rows = []
    for key, group in complete.groupby(keys[:3], dropna=False):
        record = dict(zip(keys[:3], key))
        record.update(n_seeds=len(group), final=len(group) == len(config["seeds"]))
        for metric in metrics:
            values = group[metric].dropna()
            record[metric + "_mean"] = values.mean() if len(values) else np.nan
            record[metric + "_seed_sd"] = values.std(ddof=1) if len(values) > 1 else np.nan
        summary_rows.append(record)
    summary = pd.DataFrame(summary_rows)
    summary.to_csv(root / "summary.csv", index=False)
    raw[raw.participant_count == 10].to_csv(root / "main_p10_results.csv", index=False)
    diagnostics = raw[raw.method == "NAA-GDP"]
    cols = [c for c in ("run_id", "outer_seed", "participant_count", "noise_scale", "realization",
                        "gpm_converged", "gpm_iterations", "gpm_last_step", "alignment_seconds") if c in raw]
    diagnostics[cols].to_csv(root / "alignment_diagnostics.csv", index=False)
    counts = _run_completed_counts(root)
    schedule_complete = (len(raw) == expected_result_rows(config)
                         and counts["completed_fits"] == expected_fits(config))
    if len(summary):
        _run_plots(root, summary, schedule_complete)
    noisy = raw[raw.method == "NAA-GDP"].groupby("participant_count").size().to_dict()
    report = ["# CelebA run status", "", f"Completed fitted models: {counts['completed_fits']}/{expected_fits(config)}.",
              f"Completed main result rows: {len(raw)}/{expected_result_rows(config)}.",
              f"Completed individual local fits: {counts['completed_local_fits']}/{len(config['seeds']) * max(config['participant_counts'])}.",
              f"Completed positive-noise NAA fits by participant count: {noisy}.", "",
              "All rates in CSV files are fractions. SD is across outer-seed means, after averaging",
              "four realizations within each seed and noise level. A summary row is final only when",
              "all five seeds have all required realizations. Partial figures are labelled provisional.", "",
              "The main p=10 CSV is an exact filter of raw_results.csv. No rerun or new deployment",
              "is used for the participant-count figure. The same target participant and fixed",
              "100-reference gallery pool are retained across participant counts.", "",
              "Local-only fits one MLP per participant on its own unperturbed public-SVD records.",
              "Each local model is fitted once per seed and reused in nested collaborations.",
              "Local-only result rows pool confusion counts across the first p local models, so",
              "their balanced accuracy uses the same pooled test cohort as collaboration methods.",
              "They are aggregate rows, not additional fits. Their fit_seconds is the sum of the",
              "constituent local fitting times. Privacy is unmeasured, not zero.", "",
              "This evaluates OP reconstruction followed by a frozen public face auditor; it does",
              "not establish privacy against all admissible attacks. I-GDP and Local-only have no linkage estimate."]
    (root / "REPORT.md").write_text("\n".join(report) + "\n")
    return raw


def _run_plots(root, summary, schedule_complete=False):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(6.2, 4.5))
    for method in ("NAA-GDP", "C-GDP-private-noise"):
        series = summary[(summary.method == method) & (summary.participant_count == 10)].sort_values("noise_scale")
        if len(series):
            ax.errorbar(100*series.linkage_rate_mean, 100*series.balanced_accuracy_mean,
                        xerr=100*series.linkage_rate_seed_sd.fillna(0),
                        yerr=100*series.balanced_accuracy_seed_sd.fillna(0), marker="o", capsize=2, label=method)
    for method in ("C-GDP", "AA-GDP"):
        series = summary[(summary.method == method) & (summary.participant_count == 10)]
        if len(series):
            ax.scatter(100*series.linkage_rate_mean, 100*series.balanced_accuracy_mean, label=method)
    for method, color, style in (("Local-only", "tab:green", "--"), ("I-GDP", "grey", "-.")):
        baseline = summary[(summary.method == method) & (summary.participant_count == 10)]
        if len(baseline):
            ax.axhline(100*baseline.balanced_accuracy_mean.iloc[0], linestyle=style,
                       color=color, label=method + " utility")
    ax.axvline(10, linestyle=":", color="grey", label="10-candidate chance")
    ax.set(xlabel="Frozen-auditor identity linkage (%)", ylabel="Smiling balanced accuracy (%)",
           title="CelebA, p=10" + (" (provisional)" if not schedule_complete else ""))
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(root / "privacy_utility.pdf")
    plt.close(fig)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    naa = summary[summary.method == "NAA-GDP"]
    scales = sorted(naa.noise_scale.unique())
    selected = sorted({scales[index] for index in (0, len(scales)//2, len(scales)-1)}) if scales else []
    for scale in selected:
        series = naa[naa.noise_scale == scale].sort_values("participant_count")
        axes[0].errorbar(series.participant_count, 100*series.balanced_accuracy_mean,
                         yerr=100*series.balanced_accuracy_seed_sd.fillna(0), marker=".", label=f"NAA v={scale:g}")
        axes[1].plot(series.participant_count, series.alignment_seconds_mean, marker=".", label=f"v={scale:g}")
    for method, style in (("Local-only", "--"), ("I-GDP", "-.")):
        series = summary[summary.method == method].sort_values("participant_count")
        if len(series):
            axes[0].errorbar(series.participant_count, 100*series.balanced_accuracy_mean,
                             yerr=100*series.balanced_accuracy_seed_sd.fillna(0),
                             marker="o", linestyle=style, label=method)
    axes[0].set(xlabel="Participant count p", ylabel="Pooled smiling balanced accuracy (%)")
    axes[1].set(xlabel="Participant count p", ylabel="Alignment wall time (s)")
    for ax in axes:
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize=8)
    fig.suptitle("Participant-count sensitivity" + (" (provisional)" if not schedule_complete else ""))
    fig.tight_layout()
    fig.savefig(root / "participant_scaling.pdf")
    plt.close(fig)


def expected_fits(config):
    s, r, p = len(config["seeds"]), config["realizations"], len(config["participant_counts"])
    return s * (p * (len(config["anchor_scales"]) * r + 3)
                + len(config["private_sigmas"]) * r + max(config["participant_counts"]))


def expected_result_rows(config):
    s, r, p = len(config["seeds"]), config["realizations"], len(config["participant_counts"])
    return s * (p * (len(config["anchor_scales"]) * r + 4) + len(config["private_sigmas"]) * r)


def run_experiment(config, code_hash):
    root = Path(config["output_root"])
    root.mkdir(parents=True, exist_ok=True)
    identity = {"config": config, "code_sha256": code_hash}
    signature = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
    lock = root / "protocol.json"
    if lock.exists() and json.loads(lock.read_text())["signature"] != signature:
        raise RuntimeError("Existing results use a different code/configuration. Do not mix protocols; archive deliberately before changing this experiment.")
    _run_json(lock, {**identity, "signature": signature, "expected_fits": expected_fits(config),
                     "expected_result_rows": expected_result_rows(config)})
    device = "cuda"
    if not torch.cuda.is_available():
        raise RuntimeError("Select a Colab GPU runtime before running the full experiment.")
    environment = {"python": platform.python_version(), "platform": platform.platform(),
                   "torch": torch.__version__, "cuda": torch.version.cuda,
                   "gpu": torch.cuda.get_device_name(), "packages": {
                       name: importlib.metadata.version(name) for name in
                       ("numpy", "pandas", "torch", "torchvision", "facenet-pytorch", "Pillow", "matplotlib")}}
    env_path = root / "environment.json"
    if env_path.exists():
        prior = json.loads(env_path.read_text())
        for key in ("python", "torch", "cuda", "packages"):
            if prior[key] != environment[key]:
                raise RuntimeError(f"Resume environment changed ({key}); restore the recorded package versions before continuing.")
    else:
        _run_json(env_path, environment)
    sessions_path = root / "runtime_sessions.json"
    sessions = json.loads(sessions_path.read_text()) if sessions_path.exists() else []
    sessions.append({**environment, "started_utc": datetime.now(timezone.utc).isoformat()})
    _run_json(sessions_path, sessions)
    print(f"GPU: {environment['gpu']}; scheduled fits: {expected_fits(config)}", flush=True)
    print("Algebra checks:", protocol_self_test(), flush=True)
    source = prepare_source(Path(config["old_cache"]), root / "cache")
    _run_json(root / "source_manifest.json", source["source_manifest"])
    records_dir = root / "records"
    records_dir.mkdir(exist_ok=True)
    local_dir = root / "local_models"
    local_dir.mkdir(exist_ok=True)
    start = time.perf_counter()
    deployment_identity = None
    auditor_identity = None

    def progress(status, current=None, error=None):
        counts = _run_completed_counts(root)
        payload = {"status": status, **counts, "scheduled_fits": expected_fits(config),
                   "scheduled_result_rows": expected_result_rows(config),
                   "updated_utc": datetime.now(timezone.utc).isoformat(), "current": current,
                   "error": error, "session_elapsed_seconds": time.perf_counter()-start}
        _run_json(root / "progress.json", payload)
        return counts["completed_fits"]

    def read_record(run_id, directory=None):
        path = (records_dir if directory is None else directory) / (run_id + ".json")
        if not path.exists():
            return None
        row = json.loads(path.read_text())
        if row.get("signature") != signature or row.get("source_hash") != source["source_hash"]:
            raise RuntimeError(f"Checkpoint identity mismatch: {run_id}")
        if row.get("deployment_artifact") != deployment_identity or row.get("auditor_sha256") != auditor_identity:
            raise RuntimeError(f"Checkpoint projection/deployment/auditor mismatch: {run_id}")
        return row

    try:
        # Cache only public pretrained weights in the new folder; old files stay read-only.
        old_torch = Path(config["old_cache"]) / "torch_home"
        new_torch = root / "cache" / "torch_home"
        if old_torch.is_dir():
            for src in old_torch.rglob("*.pt"):
                dst = new_torch / src.relative_to(old_torch)
                if not dst.exists():
                    dst.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copyfile(src, dst)
        auditor = FaceAuditor(device=device, torchcache=new_torch)
        auditor_identity = auditor.metadata["state_sha256"]
        auditor_path = root / "auditor.json"
        if auditor_path.exists() and json.loads(auditor_path.read_text())["state_sha256"] != auditor_identity:
            raise RuntimeError("Frozen auditor weights changed; refusing to mix evaluations.")
        _run_json(root / "auditor.json", auditor.metadata)
        for seed in config["seeds"]:
            seed_root = root / "deployments" / f"seed_{seed}"
            progress("preparing_deployment", f"seed={seed}")
            deployment = prepare_deployment(source, seed, seed_root, h=config["h"],
                                            max_participants=max(config["participant_counts"]), device=device)
            deployment_identity = {key: deployment["manifest"][key]
                                   for key in ("config_hash", "projection_sha256", "data_sha256")}
            participants = deployment["participants"]
            bank = generate_bank(seed, len(participants), config["h"], config["r"])
            clean_anchors = noisy_anchor_bank(bank, seed, 0, 0, 0)
            maps_clean = _run_clean_maps(clean_anchors, device)
            q = {split: [part[split + "_q"].numpy() for part in participants] for split in SPLITS}
            y = {split: [part[split + "_y"].numpy() for part in participants] for split in SPLITS}
            uploads = {split: [_run_matmul(block, bank["O"][i], bank["psi"][i], device)
                               for i, block in enumerate(q[split])] for split in SPLITS}
            common = {split: [_run_matmul(block, bank["O"][0], bank["psi"][0], device)
                              for block in q[split]] for split in SPLITS}
            target = config["target_participant"]
            query_upload = _run_matmul(deployment["query_q"].numpy(), bank["O"][target], bank["psi"][target], device)
            gallery = auditor.encode_gallery(deployment["gallery_images"])
            def privacy(qhat):
                result = auditor.linkage(qhat, deployment["F"], gallery,
                                         deployment["lineup_indices"], deployment["correct_positions"])
                truth = deployment["query_q"].numpy()
                result["projected_recovery_relative_error"] = float(np.linalg.norm(qhat-truth) / np.linalg.norm(truth))
                return result
            zero_privacy = privacy(op_reconstruct(bank["A"], clean_anchors[target], query_upload, device))
            training_seed = _model_seed(seed, "utility-training")
            mlp_cfg = {**config["mlp"], "device": device}

            # Local models see only their own unperturbed public-SVD records.
            # Fit each once, then pool its held-out predictions for every prefix p.
            local_rows = []
            for participant in range(len(participants)):
                local_id = f"s{seed}_i{participant}_Local-only"
                local_row = read_record(local_id, local_dir)
                if local_row is None:
                    progress("training_local", local_id)
                    beginning = time.perf_counter()
                    metrics = fit_mlp(q["train"][participant], y["train"][participant],
                                      q["val"][participant], y["val"][participant],
                                      q["test"][participant], y["test"][participant],
                                      training_seed, mlp_cfg)
                    local_row = {
                        "run_id": local_id, "signature": signature,
                        "source_hash": source["source_hash"],
                        "deployment_artifact": deployment_identity, "auditor_sha256": auditor_identity,
                        "deployment_hash": deployment["manifest"]["config_hash"],
                        "outer_seed": seed, "participant": participant, "method": "Local-only",
                        "fit_count": 1, "utility": metrics,
                        "confusion_counts": _run_local_counts(metrics, y["test"][participant]),
                        "fit_seconds": time.perf_counter() - beginning,
                        "training_seed": training_seed, "privacy": None, "linkage_rate": None,
                        "finished_utc": datetime.now(timezone.utc).isoformat(),
                    }
                    _run_json(local_dir / (local_id + ".json"), local_row)
                    count = progress("training_local", local_id)
                    print(f"[{count}/{expected_fits(config)}] {local_id}: "
                          f"BA={100*metrics['balanced_accuracy']:.2f}%, "
                          f"fit={local_row['fit_seconds']:.1f}s", flush=True)
                if local_row.get("participant") != participant or local_row.get("outer_seed") != seed:
                    raise RuntimeError(f"Local checkpoint participant/seed mismatch: {local_id}")
                local_rows.append(local_row)

            for p in config["participant_counts"]:
                aggregate_id = f"s{seed}_p{p}_Local-only_l0_r0"
                if read_record(aggregate_id) is not None:
                    continue
                selected_local = local_rows[:p]
                metrics = _run_pool_local(selected_local)
                if metrics["n_records"] != sum(len(labels) for labels in y["test"][:p]):
                    raise RuntimeError("Local aggregate does not match the collaboration test cohort.")
                row = {
                    "run_id": aggregate_id, "signature": signature, "source_hash": source["source_hash"],
                    "deployment_hash": deployment["manifest"]["config_hash"],
                    "deployment_artifact": deployment_identity, "auditor_sha256": auditor_identity,
                    "outer_seed": seed, "participant_count": p, "method": "Local-only",
                    "noise_scale": 0.0, "level_index": 0, "realization": 0,
                    "balanced_accuracy": metrics["balanced_accuracy"], "accuracy": metrics["accuracy"],
                    "utility": metrics, "linkage_rate": None, "privacy": None,
                    "fit_count": 0, "aggregate_model_count": p,
                    "constituent_local_run_ids": [item["run_id"] for item in selected_local],
                    "fit_seconds": sum(item["fit_seconds"] for item in selected_local),
                    "alignment_seconds": 0.0, "gpm_converged": None, "gpm_iterations": None,
                    "gpm_last_step": None, "alignment": None,
                    "finished_utc": datetime.now(timezone.utc).isoformat(),
                }
                _run_json(records_dir / (aggregate_id + ".json"), row)
                progress("aggregating_local", aggregate_id)
            summarize_results(root, config)

            def fit(method, p, level_index, repetition, scale, inputs, private, diag=None):
                run_id = f"s{seed}_p{p}_{method}_l{level_index}_r{repetition}"
                cached = read_record(run_id)
                if cached is not None:
                    return cached
                progress("running", run_id)
                beginning = time.perf_counter()
                labels = {split: np.concatenate(y[split][:p]) for split in SPLITS}
                metrics = fit_mlp(inputs["train"], labels["train"], inputs["val"], labels["val"],
                                  inputs["test"], labels["test"], training_seed, mlp_cfg)
                row = {"run_id": run_id, "signature": signature, "source_hash": source["source_hash"],
                       "deployment_hash": deployment["manifest"]["config_hash"],
                       "deployment_artifact": deployment_identity, "auditor_sha256": auditor_identity,
                       "outer_seed": seed, "participant_count": p, "method": method,
                       "fit_count": 1,
                       "noise_scale": float(scale), "level_index": level_index, "realization": repetition,
                       "balanced_accuracy": metrics["balanced_accuracy"], "accuracy": metrics["accuracy"],
                       "linkage_rate": private["linkage_rate"] if private else None,
                       "privacy": private, "utility": metrics,
                       "fit_seconds": time.perf_counter()-beginning,
                       "alignment_seconds": diag["elapsed_seconds"] if diag else 0.0,
                       "gpm_converged": diag["converged"] if diag else None,
                       "gpm_iterations": diag["iterations"] if diag else None,
                       "gpm_last_step": diag["last_step"] if diag else None, "alignment": diag,
                       "finished_utc": datetime.now(timezone.utc).isoformat()}
                _run_json(records_dir / (run_id + ".json"), row)
                count = progress("running", run_id)
                print(f"[{count}/{expected_fits(config)}] {run_id}: BA={100*row['balanced_accuracy']:.2f}%"
                      + (f", linkage={100*row['linkage_rate']:.2f}%" if private else "")
                      + f", fit={row['fit_seconds']:.1f}s", flush=True)
                if count % 5 == 0:
                    summarize_results(root, config)
                return row

            # Independent utility fits, identical training seed; no duplicated pseudo-repetitions.
            equivalence_errors = {}
            for p in config["participant_counts"]:
                for method, blocks, maps, leakage in (("C-GDP", common, None, zero_privacy),
                                                     ("I-GDP", uploads, None, None),
                                                     ("AA-GDP", uploads, maps_clean, zero_privacy)):
                    run_id = f"s{seed}_p{p}_{method}_l0_r0"
                    if read_record(run_id) is None:
                        inputs = _run_collaboration(blocks, maps, p, device)
                        if method == "AA-GDP":
                            baseline = np.concatenate(common["test"][:p]).astype(np.float32)
                            error = float(np.linalg.norm(inputs["test"]-baseline) / np.linalg.norm(baseline))
                            if error > 1e-6:
                                raise RuntimeError(f"Noiseless AA/C equivalence failed: {error}")
                            equivalence_errors[str(p)] = error
                        fit(method, p, 0, 0, 0, inputs, leakage)
                        del inputs
            if equivalence_errors:
                _run_json(seed_root / "noiseless_equivalence.json", equivalence_errors)

            for level, v in enumerate(config["anchor_scales"], 1):
                for repetition in range(config["realizations"]):
                    pending = [p for p in config["participant_counts"]
                               if read_record(f"s{seed}_p{p}_NAA-GDP_l{level}_r{repetition}") is None]
                    if not pending:
                        continue
                    B = noisy_anchor_bank(bank, seed, level, repetition, v)
                    # p does not enter this attack or its gallery: one evaluation is reused exactly.
                    leaked = privacy(op_reconstruct(bank["A"], B[target], query_upload, device))
                    for p in pending:
                        progress("aligning", f"seed={seed}, p={p}, v={v}, realization={repetition}")
                        maps, diag = align_anchors(B[:p], device=device, tol=config["gpm_tolerance"],
                                                   max_iters=config["gpm_max_iters"])
                        inputs = _run_collaboration(uploads, maps, p, device)
                        fit("NAA-GDP", p, level, repetition, v, inputs, leaked, diag)
                        del inputs, maps
                    del B
            # The private-noise comparator is a separate, explicitly budgeted p=10 sweep.
            p = 10
            for level, sigma in enumerate(config["private_sigmas"], 1):
                for repetition in range(config["realizations"]):
                    if read_record(f"s{seed}_p{p}_C-GDP-private-noise_l{level}_r{repetition}") is not None:
                        continue
                    noisy = {split: [block + sigma*np.random.default_rng(_model_seed(
                        seed, "private-noise", level, repetition, split, i)).standard_normal(block.shape)
                                     for i, block in enumerate(common[split][:p])] for split in SPLITS}
                    # Select the actual uploaded record, including that record's exact private noise.
                    query_y = np.stack([noisy[location["split"]][target][location["position"]]
                                        for location in deployment["query_locations"]])
                    qhat = _run_matmul(query_y-bank["psi"][0], bank["O"][0].T, np.zeros(config["h"]), device)
                    leaked = privacy(qhat)
                    inputs = _run_collaboration(noisy, None, p, device)
                    fit("C-GDP-private-noise", p, level, repetition, sigma, inputs, leaked)
                    del noisy, inputs
            summarize_results(root, config)
            del deployment, participants, q, y, uploads, common, bank, clean_anchors, maps_clean, local_rows
            gc.collect()
            torch.cuda.empty_cache()
        raw = summarize_results(root, config)
        assert len(raw) == expected_result_rows(config)
        counts = _run_completed_counts(root)
        assert counts["completed_fits"] == expected_fits(config)
        assert counts["completed_local_fits"] == len(config["seeds"]) * max(config["participant_counts"])
        assert raw[raw.method == "NAA-GDP"].groupby("participant_count").size().eq(200).all()
        progress("complete")
        print(f"Completed all {counts['completed_fits']} fitted models and {len(raw)} main result rows. "
              f"Latest results: {root}", flush=True)
        return raw
    except BaseException as exc:
        progress("interrupted" if isinstance(exc, KeyboardInterrupt) else "failed", error=str(exc))
        summarize_results(root, config)
        raise


## Run and resume
The checks run before the full schedule. Execution writes `progress.json` and one atomic JSON record per completed fit. Tables and figures refresh every five fits and at the end of each seed.


In [ ]:
# Small numerical checks exercise both spectral paths and paired-noise algebra.
print(protocol_self_test())
# Training reproducibility and validation selection, without touching CelebA.
_rng = np.random.default_rng(29)
_xx = _rng.normal(size=(120, 6)).astype(np.float32)
_yy = (_xx[:, 0] + 0.4*_xx[:, 1] > 0).astype(np.float32)
_args = (_xx[:80], _yy[:80], _xx[80:100], _yy[80:100], _xx[100:], _yy[100:], 77)
_cfg = {"device": "cuda", "max_epochs": 3, "patience": 3, "hidden_dims": [12, 6], "batch_size": 16}
_first, _second = fit_mlp(*_args, _cfg), fit_mlp(*_args, _cfg)
assert _first["training_history"] == _second["training_history"]
assert _first["balanced_accuracy"] == _second["balanced_accuracy"]
assert _first["val_best_balanced_accuracy"] == max(r["validation_balanced_accuracy"] for r in _first["training_history"])
print("MLP reproducibility and validation selection passed.")


In [ ]:
results = run_experiment(CONFIG, CODE_SHA256)


## Latest results
`raw_results.csv` contains all fits, `seed_means.csv` the within-seed averages, and `summary.csv` the mean and SD across complete seeds. Rows marked `final=False` are provisional. `main_p10_results.csv` is an exact subset of the same raw results used for scaling.


In [ ]:
display(pd.read_csv(Path(CONFIG["output_root"]) / "summary.csv"))
print((Path(CONFIG["output_root"]) / "REPORT.md").read_text())
